# 4.3 Continuous Batching Lab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/05_optimization/04.3_continuous_batching/lab.ipynb)
[![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.marimo.io/github/harshuljain13/llm-inference-at-scale/blob/master/content/05_optimization/04.3_continuous_batching/lab.ipynb)

Simulate static vs continuous batching to measure throughput, GPU utilization, and tail latency.

In [ ]:
# Cell 0: Install dependencies via subprocess (works on Colab and Molab)
import subprocess, sys
# numpy: array operations and statistics
# matplotlib: visualization of utilization and latency
subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'matplotlib']
)

In [ ]:
# Cell 1: Experiment parameters
# Change these values and re-run to explore different scenarios
#
# NUM_REQUESTS: how many inference requests to simulate
NUM_REQUESTS = 32
# MAX_BATCH_SIZE: maximum concurrent slots (GPU memory constraint)
MAX_BATCH_SIZE = 8
# MIN_GEN_LEN: shortest output any request generates
MIN_GEN_LEN = 16
# MAX_GEN_LEN: longest output any request generates
# High variance between MIN and MAX = more benefit from continuous batching
MAX_GEN_LEN = 512
# ARRIVAL_RATE: mean inter-arrival time (exponential distribution)
# Lower = more bursty arrivals, higher = more spread out
ARRIVAL_RATE = 3.0
# SEED: for reproducibility across runs
SEED = 42

In [ ]:
# Cell 2: Imports and request data structure
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import List


# Request represents a single inference request in the simulation
# It tracks the full lifecycle: arrival -> start -> completion
@dataclass
class Request:
    # unique identifier for this request
    id: int
    # number of input tokens (determines prefill cost)
    prompt_len: int
    # number of output tokens to generate
    gen_len: int
    # simulation step when this request enters the queue
    arrive_step: int
    # step when actual token generation begins (-1 = not started)
    start_step: int = -1
    # step when all tokens are generated (-1 = not finished)
    end_step: int = -1

In [ ]:
# Cell 3: Generate synthetic workload
# Creates requests with variable output lengths and staggered arrivals
# This mimics real production traffic patterns

def generate_workload(n: int = NUM_REQUESTS, seed: int = SEED) -> List[Request]:
    """Generate n requests with random output lengths and Poisson arrivals."""
    # Use numpy Generator for reproducible randomness
    rng = np.random.default_rng(seed)
    reqs = []
    for i in range(n):
        # prompt_len: uniform between 32-256 tokens
        prompt = rng.integers(32, 256)
        # gen_len: uniform between MIN and MAX (creates high variance)
        gen = rng.integers(MIN_GEN_LEN, MAX_GEN_LEN)
        # arrive_step: exponential distribution models bursty traffic
        arrive = int(rng.exponential(ARRIVAL_RATE))
        # Create request with computed values
        reqs.append(Request(id=i, prompt_len=prompt, gen_len=gen, arrive_step=arrive))
    # Sort by arrival time so simulation processes chronologically
    return sorted(reqs, key=lambda r: r.arrive_step)


# Generate the workload used throughout this notebook
requests = generate_workload()
# Compute summary statistics to understand the workload characteristics
gen_lens = [r.gen_len for r in requests]
# Print workload summary
print(f"{len(requests)} requests generated")
print(f"  Output length range: {min(gen_lens)}-{max(gen_lens)} tokens")
print(f"  Output length mean:  {np.mean(gen_lens):.0f} tokens")
print(f"  Output length std:   {np.std(gen_lens):.0f} tokens")
# High std relative to mean = high variance = continuous batching benefits most
print(f"  Coefficient of variation: {np.std(gen_lens)/np.mean(gen_lens):.2f}")

In [ ]:
# Cell 4: Static batching simulator
# Mimics traditional serving: fill batch, wait for ALL to finish, repeat

def simulate_static(requests: List[Request], max_batch: int = MAX_BATCH_SIZE):
    """Static batching: pad all requests to longest in batch."""
    # Deep copy requests so we can modify start/end without affecting originals
    reqs = [Request(**r.__dict__) for r in requests]
    # Queue holds requests waiting to be scheduled
    queue = list(reqs)
    # Current simulation timestep
    step = 0
    # Track completed requests
    completed = []
    # Track (step, utilization_fraction) for plotting
    utilization = []

    while queue:
        # Find requests that have arrived by current step
        available = [r for r in queue if r.arrive_step <= step]
        if not available:
            # No requests ready: GPU is idle this step
            utilization.append((step, 0.0))
            step += 1
            continue

        # Form a batch: take up to max_batch from available requests
        batch = available[:max_batch]
        # Remove selected requests from queue and record start time
        for r in batch:
            queue.remove(r)
            r.start_step = step

        # Static batching pads to the longest sequence in the batch
        max_gen = max(r.gen_len for r in batch)
        # Record per-step utilization during this batch
        for t in range(max_gen):
            # Count requests still generating at sub-step t
            active = sum(1 for r in batch if t < r.gen_len)
            # Utilization = active / total slots (drops as short requests finish)
            utilization.append((step + t, active / max_batch))

        # ALL requests end at same time (padded to longest)
        # Short requests waste GPU cycles waiting for long ones
        for r in batch:
            r.end_step = step + max_gen
        completed.extend(batch)
        # Advance clock past entire batch duration
        step += max_gen

    return completed, utilization

In [ ]:
# Cell 5: Continuous batching simulator
# Mimics ORCA/vLLM: scheduler runs at EVERY decode step

def simulate_continuous(requests: List[Request], max_batch: int = MAX_BATCH_SIZE):
    """Continuous batching: evict finished, admit new, every step."""
    # Deep copy to preserve originals
    reqs = [Request(**r.__dict__) for r in requests]
    # Waiting queue: requests not yet admitted to the batch
    queue = list(reqs)
    # Active batch: list of (request, tokens_generated_so_far) tuples
    active_batch: List[tuple] = []
    # Completed requests
    completed = []
    # Per-step utilization for plotting
    utilization = []
    # Current simulation timestep
    step = 0

    # Loop until all requests are processed
    while queue or active_batch:
        # PHASE 1: EVICTION
        # Remove requests that finished generating all their tokens
        still_active = []
        for r, toks in active_batch:
            if toks >= r.gen_len:
                # This request is done: record completion time
                r.end_step = step
                completed.append(r)
            else:
                # Still needs more tokens: keep in batch
                still_active.append((r, toks))
        # Update active batch to only contain unfinished requests
        active_batch = still_active

        # PHASE 2: ADMISSION
        # Fill freed slots with waiting requests (iteration-level scheduling)
        available = [r for r in queue if r.arrive_step <= step]
        while len(active_batch) < max_batch and available:
            # Admit next waiting request into a free slot
            r = available.pop(0)
            queue.remove(r)
            # Record when this request starts generating
            r.start_step = step
            # Initialize with 0 tokens generated
            active_batch.append((r, 0))

        # Record utilization: fraction of batch slots occupied
        utilization.append((step, len(active_batch) / max_batch))

        # PHASE 3: DECODE
        # Each active request generates exactly one token this step
        active_batch = [(r, toks + 1) for r, toks in active_batch]
        # Advance simulation clock
        step += 1

        # Termination: nothing left to process
        if not queue and not active_batch:
            break

    return completed, utilization

In [ ]:
# Cell 6: Run simulations and compare metrics
# Execute both strategies on the identical workload for fair comparison
static_completed, static_util = simulate_static(requests)
cont_completed, cont_util = simulate_continuous(requests)


def compute_metrics(completed: List[Request]) -> dict:
    """Compute throughput and latency from simulation results."""
    # Total tokens generated by all requests combined
    total_tokens = sum(r.gen_len for r in completed)
    # Total simulation time is when last request finishes
    total_steps = max(r.end_step for r in completed)
    # Per-request latency: time spent in system (arrival to completion)
    latencies = [r.end_step - r.arrive_step for r in completed]
    # Return all key metrics in a dict
    return {
        'throughput': total_tokens / total_steps,
        'p50_latency': np.median(latencies),
        'p99_latency': np.percentile(latencies, 99),
        'total_steps': total_steps,
    }


# Compute metrics for both strategies
static_m = compute_metrics(static_completed)
cont_m = compute_metrics(cont_completed)

# Display formatted comparison table
print("=" * 55)
print(f"{'Metric':<25} {'Static':>10} {'Continuous':>12} {'Gain':>8}")
print("=" * 55)
# Throughput: tokens processed per unit time
print(f"{'Throughput (tok/step)':<25} {static_m['throughput']:>10.1f} {cont_m['throughput']:>12.1f} {cont_m['throughput']/static_m['throughput']:>7.2f}x")
# p50: median request experiences this latency
print(f"{'p50 Latency (steps)':<25} {static_m['p50_latency']:>10.0f} {cont_m['p50_latency']:>12.0f} {static_m['p50_latency']/cont_m['p50_latency']:>7.2f}x")
# p99: tail latency, worst 1% of requests
print(f"{'p99 Latency (steps)':<25} {static_m['p99_latency']:>10.0f} {cont_m['p99_latency']:>12.0f} {static_m['p99_latency']/cont_m['p99_latency']:>7.2f}x")
# Total time to process all requests
print(f"{'Total Time (steps)':<25} {static_m['total_steps']:>10} {cont_m['total_steps']:>12} {static_m['total_steps']/cont_m['total_steps']:>7.2f}x")
print("=" * 55)

In [ ]:
# Cell 7: Plot 1 - Throughput comparison bar chart
# Visual proof that continuous batching delivers more tokens per step
fig, ax = plt.subplots(figsize=(6, 4))
# Bar chart: one bar per strategy
bars = ax.bar(
    ['Static\n(Padded)', 'Continuous\n(Iteration-level)'],
    [static_m['throughput'], cont_m['throughput']],
    # Red for wasteful static, green for efficient continuous
    color=['#ef4444', '#22c55e'],
    edgecolor='black',
    width=0.5
)
# Label axes
ax.set_ylabel('Tokens / Step')
ax.set_title('Throughput: Static vs Continuous Batching')
# Annotate bars with exact values
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, height + 0.1,
            f'{height:.1f}', ha='center', fontsize=11)
# Leave headroom for text annotations
ax.set_ylim(0, max(static_m['throughput'], cont_m['throughput']) * 1.25)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 8: Plot 2 - GPU utilization over time
# Shows the core insight: static batching has utilization drops,
# continuous batching maintains near-100% throughout
fig, axes = plt.subplots(2, 1, figsize=(12, 6))

# Top panel: static batching utilization
# Extract step numbers and utilization fractions
s_steps, s_utils = zip(*static_util)
# Shaded area shows utilization over time
axes[0].fill_between(s_steps, s_utils, alpha=0.4, color='#ef4444')
# Solid line on top of shading
axes[0].plot(s_steps, s_utils, color='#ef4444', lw=1.5)
axes[0].set_ylabel('Slot Utilization')
# Title includes computed average utilization
axes[0].set_title(f'Static Batching (avg: {np.mean(s_utils):.1%})')
# Utilization ranges from 0 to 1
axes[0].set_ylim(0, 1.05)
# Dashed line at 100% for reference
axes[0].axhline(1.0, ls='--', color='gray', lw=0.8)

# Bottom panel: continuous batching utilization
c_steps, c_utils = zip(*cont_util)
# Green shading shows high sustained utilization
axes[1].fill_between(c_steps, c_utils, alpha=0.4, color='#22c55e')
axes[1].plot(c_steps, c_utils, color='#22c55e', lw=1.5)
axes[1].set_ylabel('Slot Utilization')
axes[1].set_xlabel('Simulation Step')
axes[1].set_title(f'Continuous Batching (avg: {np.mean(c_utils):.1%})')
axes[1].set_ylim(0, 1.05)
# Same 100% reference line
axes[1].axhline(1.0, ls='--', color='gray', lw=0.8)

plt.tight_layout()
plt.show()

In [ ]:
# Cell 9: Plot 3 - Request timeline (Gantt chart)
# Each horizontal bar represents one request's time in the system
# Shorter bars = lower latency = better user experience
fig_9, axes_9 = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

# Plot both strategies side by side for visual comparison
for ax, completed, title, color in [
    (axes_9[0], sorted(static_completed, key=lambda r: r.id), 'Static', '#ef4444'),
    (axes_9[1], sorted(cont_completed, key=lambda r: r.id), 'Continuous', '#22c55e')
]:
    # Draw one horizontal bar per request
    for r in completed:
        # Bar width = processing time, left edge = start time
        ax.barh(r.id, r.end_step - r.start_step, left=r.start_step,
                color=color, alpha=0.7, edgecolor='black', linewidth=0.3)
    # Label x-axis as time
    ax.set_xlabel('Step')
    # Title identifies which strategy
    ax.set_title(f'{title} Batching')
    # Set x limit to show full timeline with margin
    ax.set_xlim(0, max(r.end_step for r in completed) * 1.05)

# Y-axis label on left panel only
axes_9[0].set_ylabel('Request ID')
# Overall figure title
fig_9.suptitle('Request Timeline (shorter bars = lower latency)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 10: Plot 4 - Latency distribution
# Histogram comparing per-request latency for both strategies
# Continuous batching should show a tighter, lower-latency distribution

# Compute latency for each request: total time in system
static_latencies = [r.end_step - r.arrive_step for r in static_completed]
cont_latencies = [r.end_step - r.arrive_step for r in cont_completed]

fig_10, ax_10 = plt.subplots(figsize=(8, 4))
# Overlapping histograms allow direct comparison
# Static: wider spread, higher tail
ax_10.hist(static_latencies, bins=20, alpha=0.6, color='#ef4444',
        label=f'Static (p50={np.median(static_latencies):.0f})')
# Continuous: tighter distribution, lower values
ax_10.hist(cont_latencies, bins=20, alpha=0.6, color='#22c55e',
        label=f'Continuous (p50={np.median(cont_latencies):.0f})')
# x-axis: time from arrival to completion
ax_10.set_xlabel('End-to-End Latency (steps)')
# y-axis: how many requests experienced this latency
ax_10.set_ylabel('Number of Requests')
ax_10.set_title('Per-Request Latency Distribution')
# Legend shows p50 for quick reference
ax_10.legend()
plt.tight_layout()
plt.show()

## Key Takeaways

| Metric | Static | Continuous | Why |
|--------|--------|------------|-----|
| Throughput | Lower (padding waste) | 2-5x higher | Slots always filled |
| GPU Utilization | Drops as short requests finish | Near-100% sustained | Immediate slot reuse |
| Tail Latency | High (short reqs wait for long) | Much lower | No padding, no blocking |

Continuous batching (ORCA, vLLM, SGLang, TGI) eliminates padding waste by treating the batch as a dynamic set:
- Finished sequences are evicted immediately at every decode step
- New requests are admitted into freed slots without waiting
- Result: near-optimal GPU utilization regardless of output length variance